## Edge AI Predictive Maintenance: Bearing Fault Detection

Data preprocessing

In [1]:
# ============================================================
# Edge AI Predictive Maintenance
# Bearing Fault Detection
#
# STEP 1: DATA PREPROCESSING
# ============================================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_PATH = "../data/raw/feature_time_48k_2048_load_1.csv"

OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("=" * 60)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 1: DATA PREPROCESSING")
print("=" * 60)

print("\nLoading dataset...")

df = pd.read_csv("feature_time_48k_2048_load_1.csv")

print("\nDataset loaded successfully.")

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


# ============================================================
# 3. DISPLAY BASIC INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("\nColumns:")

for column in df.columns:
    print(" -", column)


print("\nFirst five rows:")

print(df.head())


# ============================================================
# 4. CHECK DATA TYPES
# ============================================================

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(df.dtypes)


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = df.isnull().sum()

print(missing_values)


# ============================================================
# 6. CHECK DUPLICATE ROWS
# ============================================================

print("\n" + "=" * 60)
print("DUPLICATE ROWS")
print("=" * 60)

duplicate_count = df.duplicated().sum()

print(
    f"Number of duplicate rows: {duplicate_count}"
)


# Remove duplicates if any exist

if duplicate_count > 0:

    df = df.drop_duplicates()

    print(
        f"Removed {duplicate_count} duplicate rows."
    )

else:

    print("No duplicate rows found.")


# ============================================================
# 7. IDENTIFY TARGET COLUMN
# ============================================================

TARGET_COLUMN = "fault"

print("\nTarget column:", TARGET_COLUMN)


if TARGET_COLUMN not in df.columns:

    raise ValueError(
        f"Target column '{TARGET_COLUMN}' "
        "was not found in the dataset."
    )


# ============================================================
# 8. CHECK CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("FAULT CLASS DISTRIBUTION")
print("=" * 60)

class_distribution = df[TARGET_COLUMN].value_counts()

print(class_distribution)


# ============================================================
# 9. REMOVE INVALID NUMERICAL VALUES
# ============================================================

feature_columns = [
    column
    for column in df.columns
    if column != TARGET_COLUMN
]


print("\nFeature columns:")

for column in feature_columns:
    print(" -", column)


# Convert feature columns to numeric

for column in feature_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# Replace infinity with NaN

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# Remove rows containing missing values

before_cleaning = len(df)

df = df.dropna()

after_cleaning = len(df)

removed_rows = (
    before_cleaning -
    after_cleaning
)

print(
    f"\nRows removed during cleaning: "
    f"{removed_rows}"
)


# ============================================================
# 10. SEPARATE FEATURES AND TARGET
# ============================================================

X = df[feature_columns]

y = df[TARGET_COLUMN]


print("\n" + "=" * 60)
print("FEATURE / TARGET SHAPE")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 11. ENCODE TARGET LABELS
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)


print("\n" + "=" * 60)
print("LABEL ENCODING")
print("=" * 60)

print("\nOriginal labels:")

print(
    label_encoder.classes_
)


print("\nEncoded labels:")

for original, encoded in zip(
    label_encoder.classes_,
    range(len(label_encoder.classes_))
):

    print(
        f"{original} -> {encoded}"
    )


# ============================================================
# 12. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.20,

    random_state=42,

    stratify=y_encoded
)


print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(
    "Training samples:",
    X_train.shape[0]
)

print(
    "Testing samples:",
    X_test.shape[0]
)


# ============================================================
# 13. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


print("\nFeature scaling completed.")


# ============================================================
# 14. SAVE PROCESSED DATA
# ============================================================

np.save(
    f"{OUTPUT_DIR}/X_train.npy",
    X_train_scaled
)

np.save(
    f"{OUTPUT_DIR}/X_test.npy",
    X_test_scaled
)

np.save(
    f"{OUTPUT_DIR}/y_train.npy",
    y_train
)

np.save(
    f"{OUTPUT_DIR}/y_test.npy",
    y_test
)


# Save feature names

feature_info = pd.DataFrame({
    "feature": feature_columns
})

feature_info.to_csv(
    f"{OUTPUT_DIR}/feature_names.csv",
    index=False
)


# Save label mapping

label_mapping = pd.DataFrame({

    "encoded_label":
        range(len(label_encoder.classes_)),

    "original_label":
        label_encoder.classes_

})

label_mapping.to_csv(
    f"{OUTPUT_DIR}/label_mapping.csv",
    index=False
)


# Save scaler and encoder

joblib.dump(
    scaler,
    f"{OUTPUT_DIR}/scaler.pkl"
)

joblib.dump(
    label_encoder,
    f"{OUTPUT_DIR}/label_encoder.pkl"
)


# ============================================================
# 15. SAVE CLEAN DATASET
# ============================================================

clean_dataset_path = (
    f"{OUTPUT_DIR}/clean_bearing_features.csv"
)

df.to_csv(
    clean_dataset_path,
    index=False
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("STEP 1 COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    f"\nFinal dataset size: "
    f"{df.shape[0]} rows × {df.shape[1]} columns"
)

print(
    f"\nNumber of features: "
    f"{len(feature_columns)}"
)

print(
    f"Number of fault classes: "
    f"{len(label_encoder.classes_)}"
)

print("\nFault classes:")

for class_name in label_encoder.classes_:

    print(
        f" - {class_name}"
    )


print("\nGenerated files:")

print(
    " - data/processed/X_train.npy"
)

print(
    " - data/processed/X_test.npy"
)

print(
    " - data/processed/y_train.npy"
)

print(
    " - data/processed/y_test.npy"
)

print(
    " - data/processed/feature_names.csv"
)

print(
    " - data/processed/label_mapping.csv"
)

print(
    " - data/processed/scaler.pkl"
)

print(
    " - data/processed/label_encoder.pkl"
)

print(
    " - data/processed/clean_bearing_features.csv"
)

EDGE AI BEARING FAULT DETECTION
STEP 1: DATA PREPROCESSING

Loading dataset...

Dataset loaded successfully.
Rows    : 2300
Columns : 10

DATASET INFORMATION

Columns:
 - max
 - min
 - mean
 - sd
 - rms
 - skewness
 - kurtosis
 - crest
 - form
 - fault

First five rows:
       max      min      mean        sd       rms  skewness  kurtosis  \
0  0.35986 -0.41890  0.017840  0.122746  0.124006 -0.118571 -0.042219   
1  0.46772 -0.36111  0.022255  0.132488  0.134312  0.174699 -0.081548   
2  0.46855 -0.43809  0.020470  0.149651  0.151008  0.040339 -0.274069   
3  0.58475 -0.54303  0.020960  0.157067  0.158422 -0.023266  0.134692   
4  0.44685 -0.57891  0.022167  0.138189  0.139922 -0.081534  0.402783   

      crest      form       fault  
0  2.901946  6.950855  Ball_007_1  
1  3.482334  6.035202  Ball_007_1  
2  3.102819  7.376926  Ball_007_1  
3  3.691097  7.558387  Ball_007_1  
4  3.193561  6.312085  Ball_007_1  

DATA TYPES
max         float64
min         float64
mean        float64
sd

In [2]:
from pathlib import Path
import os
import glob

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path("/workspaces/edge-ai-bearing-fault-detection")

DATA_DIR = PROJECT_ROOT / "data" / "raw"

OUTPUT_DIR = PROJECT_ROOT / "results" / "signal_processing"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project root: /workspaces/edge-ai-bearing-fault-detection
Data directory: /workspaces/edge-ai-bearing-fault-detection/data/raw
Data directory exists: True


In [3]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/workspaces/edge-ai-bearing-fault-detection"
)

DATA_DIR = PROJECT_ROOT / "raw"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "signal_processing"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Data directory:", DATA_DIR)
print("Folder exists:", DATA_DIR.exists())

mat_files = list(DATA_DIR.glob("*.mat"))

print("MAT files found:", len(mat_files))

for file in mat_files:
    print(" -", file.name)

Data directory: /workspaces/edge-ai-bearing-fault-detection/raw
Folder exists: True
MAT files found: 10
 - IR021_1_214.mat
 - B021_1_227.mat
 - IR007_1_110.mat
 - OR021_6_1_239.mat
 - OR014_6_1_202.mat
 - IR014_1_175.mat
 - B007_1_123.mat
 - B014_1_190.mat
 - Time_Normal_1_098.mat
 - OR007_6_1_136.mat


In [4]:
# ============================================================
# Edge AI Predictive Maintenance
# Bearing Fault Detection
#
# STEP 3: FEATURE ENGINEERING
# File: 03_feature_engineering.py
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_ROOT = "/workspaces/edge-ai-bearing-fault-detection"

# Your CSV is currently in the project root
DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "feature_time_48k_2048_load_1.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("=" * 70)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 3: FEATURE ENGINEERING")
print("=" * 70)

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print(
    f"Dataset shape: {df.shape}"
)


# ============================================================
# 3. DISPLAY ORIGINAL FEATURES
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL FEATURES")
print("=" * 70)

print(
    df.columns.tolist()
)


# ============================================================
# 4. BASIC DATA CLEANING
# ============================================================

print("\n" + "=" * 70)
print("DATA CLEANING")
print("=" * 70)

# Replace infinite values

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows containing missing values

missing_before = df.isnull().sum().sum()

print(
    "Missing values before cleaning:",
    missing_before
)

df = df.dropna()

missing_after = df.isnull().sum().sum()

print(
    "Missing values after cleaning:",
    missing_after
)


# ============================================================
# 5. REMOVE IDENTIFIER COLUMN
# ============================================================

# 'id' identifies the sample but should not be used
# as a predictive feature.

if "id" in df.columns:

    df = df.drop(
        columns=["id"]
    )

    print(
        "\nRemoved identifier column: id"
    )


# ============================================================
# 6. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "max",
    "min",
    "mean",
    "sd",
    "rms",
    "skewness",
    "kurtosis",
    "crest",
    "form",
    "fault"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )


# ============================================================
# 7. ORIGINAL FEATURE STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL FEATURE STATISTICS")
print("=" * 70)

print(
    df[
        [
            "max",
            "min",
            "mean",
            "sd",
            "rms",
            "skewness",
            "kurtosis",
            "crest",
            "form"
        ]
    ].describe().round(4)
)


# ============================================================
# 8. DOMAIN-SPECIFIC FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 70)
print("CREATING ENGINEERED FEATURES")
print("=" * 70)


# ------------------------------------------------------------
# 8.1 Peak-to-Peak Amplitude
# ------------------------------------------------------------

df["peak_to_peak"] = (
    df["max"] - df["min"]
)


# ------------------------------------------------------------
# 8.2 Absolute Peak
# ------------------------------------------------------------

df["absolute_peak"] = (
    np.maximum(
        np.abs(df["max"]),
        np.abs(df["min"])
    )
)


# ------------------------------------------------------------
# 8.3 Variance
# ------------------------------------------------------------

df["variance"] = (
    df["sd"] ** 2
)


# ------------------------------------------------------------
# 8.4 RMS-to-Standard-Deviation Ratio
# ------------------------------------------------------------

df["rms_sd_ratio"] = (
    df["rms"] /
    (df["sd"] + 1e-10)
)


# ------------------------------------------------------------
# 8.5 Peak-to-RMS Ratio
# ------------------------------------------------------------

df["peak_rms_ratio"] = (
    df["absolute_peak"] /
    (df["rms"] + 1e-10)
)


# ------------------------------------------------------------
# 8.6 Peak-to-Mean Ratio
# ------------------------------------------------------------

df["peak_mean_ratio"] = (
    df["absolute_peak"] /
    (np.abs(df["mean"]) + 1e-10)
)


# ------------------------------------------------------------
# 8.7 RMS Squared / Signal Energy Proxy
# ------------------------------------------------------------

df["rms_energy"] = (
    df["rms"] ** 2
)


# ------------------------------------------------------------
# 8.8 Amplitude Range Relative to RMS
# ------------------------------------------------------------

df["range_rms_ratio"] = (
    df["peak_to_peak"] /
    (df["rms"] + 1e-10)
)


# ------------------------------------------------------------
# 8.9 Kurtosis × RMS
# ------------------------------------------------------------

df["kurtosis_rms"] = (
    df["kurtosis"] *
    df["rms"]
)


# ------------------------------------------------------------
# 8.10 Skewness × RMS
# ------------------------------------------------------------

df["skewness_rms"] = (
    df["skewness"] *
    df["rms"]
)


# ============================================================
# 9. LOG TRANSFORM HIGHLY SKEWED FEATURES
# ============================================================

# Log transformations can reduce the effect of very large
# values while preserving their relative differences.

log_features = [
    "rms",
    "sd",
    "absolute_peak",
    "peak_to_peak",
    "variance",
    "rms_energy"
]

for feature in log_features:

    df[f"log_{feature}"] = np.log1p(
        np.abs(df[feature])
    )


# ============================================================
# 10. CHECK FOR INVALID VALUES
# ============================================================

print("\nChecking engineered features...")

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

invalid_rows = df.isnull().any(
    axis=1
).sum()

print(
    "Rows containing invalid values:",
    invalid_rows
)

if invalid_rows > 0:

    df = df.dropna()

    print(
        "Invalid rows removed."
    )


# ============================================================
# 11. IDENTIFY FEATURE COLUMNS
# ============================================================

TARGET_COLUMN = "fault"

feature_columns = [
    column
    for column in df.columns
    if column != TARGET_COLUMN
]


print("\n" + "=" * 70)
print("ENGINEERED FEATURE LIST")
print("=" * 70)

for index, feature in enumerate(
    feature_columns,
    start=1
):

    print(
        f"{index:02d}. {feature}"
    )


# ============================================================
# 12. CHECK FEATURE CORRELATION
# ============================================================

print("\n" + "=" * 70)
print("CORRELATION CHECK")
print("=" * 70)

correlation_matrix = df[
    feature_columns
].corr()

# Find highly correlated feature pairs

high_correlation_pairs = []

for i in range(
    len(feature_columns)
):

    for j in range(
        i + 1,
        len(feature_columns)
    ):

        feature_1 = (
            feature_columns[i]
        )

        feature_2 = (
            feature_columns[j]
        )

        correlation = (
            correlation_matrix.loc[
                feature_1,
                feature_2
            ]
        )

        if abs(correlation) >= 0.95:

            high_correlation_pairs.append(
                (
                    feature_1,
                    feature_2,
                    correlation
                )
            )


print(
    f"Highly correlated pairs "
    f"(>|0.95|): "
    f"{len(high_correlation_pairs)}"
)


for feature_1, feature_2, correlation in (
    high_correlation_pairs
):

    print(
        f"{feature_1} <-> "
        f"{feature_2}: "
        f"{correlation:.4f}"
    )


# ============================================================
# 13. SAVE CORRELATION MATRIX
# ============================================================

correlation_path = os.path.join(
    OUTPUT_DIR,
    "feature_correlation_matrix.csv"
)

correlation_matrix.to_csv(
    correlation_path
)

print(
    "\nSaved correlation matrix:"
)

print(
    correlation_path
)


# ============================================================
# 14. CREATE FINAL FEATURE DATASET
# ============================================================

output_path = os.path.join(
    OUTPUT_DIR,
    "engineered_bearing_features.csv"
)

df.to_csv(
    output_path,
    index=False
)

print(
    "\nSaved engineered dataset:"
)

print(
    output_path
)


# ============================================================
# 15. SAVE FEATURE NAMES
# ============================================================

feature_names_path = os.path.join(
    OUTPUT_DIR,
    "engineered_feature_names.csv"
)

pd.DataFrame({
    "feature_name": feature_columns
}).to_csv(
    feature_names_path,
    index=False
)

print(
    "Saved feature names:"
)

print(
    feature_names_path
)


# ============================================================
# 16. FEATURE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)

print(
    "Original rows:",
    len(pd.read_csv(DATA_PATH))
)

print(
    "Final rows:",
    len(df)
)

print(
    "Total features:",
    len(feature_columns)
)

print(
    "Target column:",
    TARGET_COLUMN
)

print(
    "Number of fault classes:",
    df[TARGET_COLUMN].nunique()
)


# ============================================================
# 17. CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("FAULT CLASS DISTRIBUTION")
print("=" * 70)

class_distribution = (
    df[TARGET_COLUMN]
    .value_counts()
)

print(
    class_distribution
)


# ============================================================
# 18. FINAL FEATURE DATASET PREVIEW
# ============================================================

print("\n" + "=" * 70)
print("ENGINEERED DATASET PREVIEW")
print("=" * 70)

print(
    df.head()
)


# ============================================================
# 19. COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nGenerated files:"
)

print(
    "1. engineered_bearing_features.csv"
)

print(
    "2. engineered_feature_names.csv"
)

print(
    "3. feature_correlation_matrix.csv"
)

print(
    "\nNext step:"
)

print(
    "STEP 4 - Machine Learning Model Development"
)

EDGE AI BEARING FAULT DETECTION
STEP 3: FEATURE ENGINEERING

Loading dataset...
Dataset shape: (2300, 10)

ORIGINAL FEATURES
['max', 'min', 'mean', 'sd', 'rms', 'skewness', 'kurtosis', 'crest', 'form', 'fault']

DATA CLEANING
Missing values before cleaning: 0
Missing values after cleaning: 0

ORIGINAL FEATURE STATISTICS


             max        min       mean         sd        rms   skewness  \
count  2300.0000  2300.0000  2300.0000  2300.0000  2300.0000  2300.0000   
mean      1.5751    -1.5510     0.0157     0.3416     0.3423    -0.0423   
std       1.5784     1.6027     0.0065     0.3053     0.3048     0.1808   
min       0.1573    -6.2926     0.0032     0.0591     0.0611    -1.0899   
25%       0.4564    -2.1750     0.0112     0.1355     0.1364    -0.1034   
50%       0.7945    -0.7337     0.0137     0.1886     0.1907    -0.0025   
75%       2.2784    -0.4270     0.0186     0.5556     0.5557     0.0611   
max       6.8259    -0.1602     0.0384     1.2566     1.2563     1.0595   

        kurtosis      crest       form  
count  2300.0000  2300.0000  2300.0000  
mean      2.6644     4.1731    26.5448  
std       4.4111     1.1483    29.2097  
min      -0.8038     2.4285     3.4844  
25%      -0.0152     3.2604     7.4134  
50%       0.8170     3.9216    13.1228  
75%       3.9023     4.8159    39.911

In [5]:
# ============================================================
# EDGE AI BEARING FAULT DETECTION
# STEP 3: FEATURE ENGINEERING
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = "/workspaces/edge-ai-bearing-fault-detection"

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "feature_time_48k_2048_load_1.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 65)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 3: FEATURE ENGINEERING")
print("=" * 65)

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


# ============================================================
# 3. DISPLAY ORIGINAL COLUMNS
# ============================================================

print("\nOriginal columns:")

for column in df.columns:
    print(" -", column)


# ============================================================
# 4. CLEAN DATA
# ============================================================

print("\n" + "=" * 65)
print("DATA CLEANING")
print("=" * 65)

# Replace infinity values
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Missing values:",
    df.isnull().sum().sum()
)

# Remove missing rows
df = df.dropna()

print(
    "Shape after cleaning:",
    df.shape
)


# ============================================================
# 5. REMOVE ID COLUMN
# ============================================================

if "id" in df.columns:

    df = df.drop(
        columns=["id"]
    )

    print(
        "\nRemoved ID column."
    )


# ============================================================
# 6. DEFINE TARGET
# ============================================================

TARGET = "fault"

if TARGET not in df.columns:

    raise ValueError(
        "Target column 'fault' was not found."
    )


# ============================================================
# 7. ORIGINAL NUMERICAL FEATURES
# ============================================================

base_features = [
    "max",
    "min",
    "mean",
    "sd",
    "rms",
    "skewness",
    "kurtosis",
    "crest",
    "form"
]

print("\nBase features:")

for feature in base_features:
    print(" -", feature)


# ============================================================
# 8. CREATE NEW FEATURES
# ============================================================

print("\n" + "=" * 65)
print("CREATING ENGINEERED FEATURES")
print("=" * 65)


# Peak-to-peak amplitude
df["peak_to_peak"] = (
    df["max"] - df["min"]
)


# Absolute peak
df["absolute_peak"] = np.maximum(
    np.abs(df["max"]),
    np.abs(df["min"])
)


# Variance
df["variance"] = (
    df["sd"] ** 2
)


# RMS / standard deviation
df["rms_sd_ratio"] = (
    df["rms"] /
    (df["sd"] + 1e-10)
)


# Peak / RMS
df["peak_rms_ratio"] = (
    df["absolute_peak"] /
    (df["rms"] + 1e-10)
)


# Peak / mean
df["peak_mean_ratio"] = (
    df["absolute_peak"] /
    (np.abs(df["mean"]) + 1e-10)
)


# RMS energy proxy
df["rms_energy"] = (
    df["rms"] ** 2
)


# Peak-to-peak / RMS
df["range_rms_ratio"] = (
    df["peak_to_peak"] /
    (df["rms"] + 1e-10)
)


# Kurtosis × RMS
df["kurtosis_rms"] = (
    df["kurtosis"] *
    df["rms"]
)


# Skewness × RMS
df["skewness_rms"] = (
    df["skewness"] *
    df["rms"]
)


# ============================================================
# 9. CREATE LOG FEATURES
# ============================================================

log_features = [
    "rms",
    "sd",
    "absolute_peak",
    "peak_to_peak",
    "variance",
    "rms_energy"
]

for feature in log_features:

    df[f"log_{feature}"] = np.log1p(
        np.abs(df[feature])
    )


# ============================================================
# 10. CHECK ENGINEERED FEATURES
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna()

feature_columns = [
    column
    for column in df.columns
    if column != TARGET
]


print("\nTotal features:", len(feature_columns))

print("\nFinal feature list:")

for number, feature in enumerate(
    feature_columns,
    start=1
):

    print(
        f"{number:02d}. {feature}"
    )


# ============================================================
# 11. CORRELATION ANALYSIS
# ============================================================

print("\n" + "=" * 65)
print("CORRELATION ANALYSIS")
print("=" * 65)

correlation_matrix = df[
    feature_columns
].corr()

high_corr_pairs = []

for i in range(
    len(feature_columns)
):

    for j in range(
        i + 1,
        len(feature_columns)
    ):

        correlation = correlation_matrix.iloc[
            i, j
        ]

        if abs(correlation) >= 0.95:

            high_corr_pairs.append(
                (
                    feature_columns[i],
                    feature_columns[j],
                    correlation
                )
            )


print(
    "Highly correlated pairs:",
    len(high_corr_pairs)
)

for feature1, feature2, corr in high_corr_pairs:

    print(
        f"{feature1} <-> {feature2}: "
        f"{corr:.4f}"
    )


# ============================================================
# 12. SAVE ENGINEERED DATASET
# ============================================================

output_dataset = os.path.join(
    OUTPUT_DIR,
    "engineered_bearing_features.csv"
)

df.to_csv(
    output_dataset,
    index=False
)

print(
    "\nSaved:",
    output_dataset
)


# ============================================================
# 13. SAVE FEATURE NAMES
# ============================================================

feature_names = pd.DataFrame({
    "feature": feature_columns
})

feature_names_path = os.path.join(
    OUTPUT_DIR,
    "engineered_feature_names.csv"
)

feature_names.to_csv(
    feature_names_path,
    index=False
)

print(
    "Saved:",
    feature_names_path
)


# ============================================================
# 14. SAVE CORRELATION MATRIX
# ============================================================

correlation_path = os.path.join(
    OUTPUT_DIR,
    "feature_correlation_matrix.csv"
)

correlation_matrix.to_csv(
    correlation_path
)

print(
    "Saved:",
    correlation_path
)


# ============================================================
# 15. CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 65)
print("FAULT CLASS DISTRIBUTION")
print("=" * 65)

print(
    df[TARGET].value_counts()
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("STEP 3 COMPLETED SUCCESSFULLY")
print("=" * 65)

print(
    "\nRows:",
    df.shape[0]
)

print(
    "Features:",
    len(feature_columns)
)

print(
    "Target:",
    TARGET
)

print(
    "Fault classes:",
    df[TARGET].nunique()
)

print("\nGenerated files:")

print(
    "1. engineered_bearing_features.csv"
)

print(
    "2. engineered_feature_names.csv"
)

print(
    "3. feature_correlation_matrix.csv"
)

print("\nNext step: STEP 4 - ML MODEL DEVELOPMENT")

EDGE AI BEARING FAULT DETECTION
STEP 3: FEATURE ENGINEERING

Loading dataset...
Dataset shape: (2300, 10)

Original columns:
 - max
 - min
 - mean
 - sd
 - rms
 - skewness
 - kurtosis
 - crest
 - form
 - fault

DATA CLEANING
Missing values: 0
Shape after cleaning: (2300, 10)

Base features:
 - max
 - min
 - mean
 - sd
 - rms
 - skewness
 - kurtosis
 - crest
 - form

CREATING ENGINEERED FEATURES

Total features: 25

Final feature list:
01. max
02. min
03. mean
04. sd
05. rms
06. skewness
07. kurtosis
08. crest
09. form
10. peak_to_peak
11. absolute_peak
12. variance
13. rms_sd_ratio
14. peak_rms_ratio
15. peak_mean_ratio
16. rms_energy
17. range_rms_ratio
18. kurtosis_rms
19. skewness_rms
20. log_rms
21. log_sd
22. log_absolute_peak
23. log_peak_to_peak
24. log_variance
25. log_rms_energy

CORRELATION ANALYSIS
Highly correlated pairs: 57
max <-> min: -0.9928
max <-> sd: 0.9517
max <-> rms: 0.9517
max <-> peak_to_peak: 0.9982
max <-> absolute_peak: 0.9970
max <-> log_rms: 0.9551
max <-> 

In [6]:
# ============================================================
# EDGE AI BEARING FAULT DETECTION
# STEP 4: MACHINE LEARNING MODEL DEVELOPMENT
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = (
    "/workspaces/edge-ai-bearing-fault-detection"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "engineered_bearing_features.csv"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "results",
    "ml_models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("=" * 70)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 4: MACHINE LEARNING MODEL DEVELOPMENT")
print("=" * 70)

print("\nLoading engineered dataset...")

df = pd.read_csv(DATA_PATH)

print(
    "Dataset shape:",
    df.shape
)


# ============================================================
# 3. DEFINE FEATURES AND TARGET
# ============================================================

TARGET = "fault"

if TARGET not in df.columns:

    raise ValueError(
        "Target column 'fault' not found."
    )


X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


print(
    "\nNumber of features:",
    X.shape[1]
)

print(
    "Number of samples:",
    X.shape[0]
)

print(
    "Number of classes:",
    y.nunique()
)


# ============================================================
# 4. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(
    y
)

print("\nFault classes:")

for encoded, original in enumerate(
    label_encoder.classes_
):

    print(
        f"{encoded} -> {original}"
    )


# Save encoder

encoder_path = os.path.join(
    MODEL_DIR,
    "label_encoder.pkl"
)

joblib.dump(
    label_encoder,
    encoder_path
)


# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        random_state=42,
        stratify=y_encoded
    )
)


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)


# ============================================================
# 6. DEFINE MODELS
# ============================================================

models = {

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            SVC(
                kernel="rbf",
                C=10,
                gamma="scale"
            )
        )
    ]),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


# ============================================================
# 7. TRAIN MODELS
# ============================================================

results = []

trained_models = {}


for model_name, model in models.items():

    print("\n" + "=" * 70)

    print(
        f"TRAINING: {model_name}"
    )

    print("=" * 70)

    # Train
    model.fit(
        X_train,
        y_train
    )

    # Predict
    y_pred = model.predict(
        X_test
    )

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # Store
    results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1_Score": f1

    })

    trained_models[
        model_name
    ] = model

    # Print metrics
    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    # ========================================================
    # Classification Report
    # ========================================================

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )

    # ========================================================
    # Confusion Matrix
    # ========================================================

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_encoder.classes_
    )

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    display.plot(
        ax=ax,
        xticks_rotation=45
    )

    plt.title(
        f"Confusion Matrix - {model_name}"
    )

    plt.tight_layout()

    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
    )

    confusion_path = os.path.join(
        RESULT_DIR,
        f"{safe_name}_confusion_matrix.png"
    )

    plt.savefig(
        confusion_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved:",
        confusion_path
    )

    # ========================================================
    # Save Model
    # ========================================================

    model_path = os.path.join(
        MODEL_DIR,
        f"{safe_name}.pkl"
    )

    joblib.dump(
        model,
        model_path
    )

    print(
        "Saved model:",
        model_path
    )


# ============================================================
# 8. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 9. SAVE MODEL COMPARISON
# ============================================================

comparison_path = os.path.join(
    RESULT_DIR,
    "model_comparison.csv"
)

results_df.to_csv(
    comparison_path,
    index=False
)

print(
    "\nSaved:",
    comparison_path
)


# ============================================================
# 10. MODEL PERFORMANCE PLOT
# ============================================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1_Score"
]

x = np.arange(
    len(results_df["Model"])
)

width = 0.18

plt.figure(
    figsize=(13, 7)
)

for index, metric in enumerate(
    metrics
):

    plt.bar(
        x + index * width,
        results_df[metric],
        width,
        label=metric
    )


plt.xticks(
    x + width * 1.5,
    results_df["Model"],
    rotation=20
)

plt.ylabel(
    "Score"
)

plt.ylim(
    0,
    1.05
)

plt.title(
    "Machine Learning Model Performance"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

performance_path = os.path.join(
    RESULT_DIR,
    "model_performance_comparison.png"
)

plt.savefig(
    performance_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    "Saved:",
    performance_path
)


# ============================================================
# 11. SAVE TRAIN / TEST DATA
# ============================================================

X_train.to_csv(
    os.path.join(
        RESULT_DIR,
        "X_train.csv"
    ),
    index=False
)

X_test.to_csv(
    os.path.join(
        RESULT_DIR,
        "X_test.csv"
    ),
    index=False
)

pd.DataFrame(
    {"target": y_train}
).to_csv(
    os.path.join(
        RESULT_DIR,
        "y_train.csv"
    ),
    index=False
)

pd.DataFrame(
    {"target": y_test}
).to_csv(
    os.path.join(
        RESULT_DIR,
        "y_test.csv"
    ),
    index=False
)


# ============================================================
# 12. SAVE TEST PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({
    "actual": label_encoder.inverse_transform(
        y_test
    )
})


# Add predictions from each model

for model_name, model in trained_models.items():

    y_pred = model.predict(
        X_test
    )

    prediction_df[
        model_name
    ] = label_encoder.inverse_transform(
        y_pred
    )


prediction_path = os.path.join(
    RESULT_DIR,
    "model_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

print(
    "Saved:",
    prediction_path
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nModels trained:",
    len(models)
)

for model_name in models:

    print(
        " -",
        model_name
    )

print(
    "\nGenerated directories:"
)

print(
    "models/"
)

print(
    "results/ml_models/"
)

print(
    "\nGenerated files include:"
)

print(
    " - model_comparison.csv"
)

print(
    " - model_performance_comparison.png"
)

print(
    " - confusion matrices"
)

print(
    " - model_predictions.csv"
)

print(
    " - trained .pkl models"
)

print(
    "\nNext step:"
)

print(
    "STEP 5 - Hyperparameter Optimization & Feature Selection"
)

EDGE AI BEARING FAULT DETECTION
STEP 4: MACHINE LEARNING MODEL DEVELOPMENT

Loading engineered dataset...
Dataset shape: (2300, 26)

Number of features: 25
Number of samples: 2300
Number of classes: 10

Fault classes:
0 -> Ball_007_1
1 -> Ball_014_1
2 -> Ball_021_1
3 -> IR_007_1
4 -> IR_014_1
5 -> IR_021_1
6 -> Normal_1
7 -> OR_007_6_1
8 -> OR_014_6_1
9 -> OR_021_6_1

TRAIN / TEST SPLIT
Training samples: 1840
Testing samples: 460

TRAINING: Logistic Regression
Accuracy : 0.9261
Precision: 0.9316
Recall   : 0.9261
F1 Score : 0.9267

Classification Report:
              precision    recall  f1-score   support

  Ball_007_1       0.95      0.89      0.92        46
  Ball_014_1       0.97      0.78      0.87        46
  Ball_021_1       0.84      0.78      0.81        46
    IR_007_1       0.96      1.00      0.98        46
    IR_014_1       1.00      1.00      1.00        46
    IR_021_1       0.92      1.00      0.96        46
    Normal_1       1.00      1.00      1.00        46
  OR_0

In [7]:
# ============================================================
# EDGE AI BEARING FAULT DETECTION
# STEP 5: HYPERPARAMETER OPTIMIZATION & FEATURE SELECTION
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.feature_selection import (
    SelectKBest,
    f_classif
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = (
    "/workspaces/edge-ai-bearing-fault-detection"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "engineered_bearing_features.csv"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "results",
    "optimization"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 2. HEADER
# ============================================================

print("=" * 70)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 5: HYPERPARAMETER OPTIMIZATION")
print("=" * 70)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading engineered features...")

df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape
)


# ============================================================
# 4. FEATURES / TARGET
# ============================================================

TARGET = "fault"

X = df.drop(
    columns=[TARGET]
)

y = df[TARGET]


# ============================================================
# 5. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(
    y
)

print(
    "\nNumber of classes:",
    len(label_encoder.classes_)
)

print(
    "Classes:",
    list(label_encoder.classes_)
)


# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        random_state=42,
        stratify=y_encoded
    )
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# 7. FEATURE SELECTION
# ============================================================

print("\n" + "=" * 70)
print("FEATURE SELECTION")
print("=" * 70)

# ANOVA F-test

selector = SelectKBest(
    score_func=f_classif,
    k="all"
)

selector.fit(
    X_train,
    y_train
)

feature_scores = pd.DataFrame({
    "feature": X_train.columns,
    "f_score": selector.scores_,
    "p_value": selector.pvalues_
})

feature_scores = feature_scores.sort_values(
    by="f_score",
    ascending=False
)

print("\nFeature ranking:")

print(
    feature_scores.to_string(
        index=False
    )
)


# Save feature scores

feature_score_path = os.path.join(
    RESULT_DIR,
    "feature_scores.csv"
)

feature_scores.to_csv(
    feature_score_path,
    index=False
)

print(
    "\nSaved:",
    feature_score_path
)


# ============================================================
# 8. SELECT TOP FEATURES
# ============================================================

TOP_K = min(
    15,
    X_train.shape[1]
)

selected_features = (
    feature_scores
    .head(TOP_K)
    ["feature"]
    .tolist()
)

print(
    f"\nSelected top {TOP_K} features:"
)

for feature in selected_features:
    print(
        " -",
        feature
    )


X_train_selected = X_train[
    selected_features
]

X_test_selected = X_test[
    selected_features
]


# Save selected feature list

selected_features_path = os.path.join(
    RESULT_DIR,
    "selected_features.csv"
)

pd.DataFrame({
    "feature": selected_features
}).to_csv(
    selected_features_path,
    index=False
)


# ============================================================
# 9. RANDOM FOREST OPTIMIZATION
# ============================================================

print("\n" + "=" * 70)
print("RANDOM FOREST OPTIMIZATION")
print("=" * 70)

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_params = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "max_depth": [
        None,
        10,
        20
    ],

    "min_samples_split": [
        2,
        5
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}


rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(
    X_train_selected,
    y_train
)

best_rf = rf_grid.best_estimator_

print(
    "\nBest Random Forest parameters:"
)

print(
    rf_grid.best_params_
)

print(
    "Best CV F1:",
    round(
        rf_grid.best_score_,
        4
    )
)


# ============================================================
# 10. SVM OPTIMIZATION
# ============================================================

print("\n" + "=" * 70)
print("SVM OPTIMIZATION")
print("=" * 70)

svm_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "svm",
        SVC()
    )
])


svm_params = {

    "svm__C": [
        0.1,
        1,
        10,
        100
    ],

    "svm__gamma": [
        "scale",
        0.01,
        0.1
    ],

    "svm__kernel": [
        "rbf"
    ]
}


svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_params,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(
    X_train_selected,
    y_train
)

best_svm = svm_grid.best_estimator_

print(
    "\nBest SVM parameters:"
)

print(
    svm_grid.best_params_
)

print(
    "Best CV F1:",
    round(
        svm_grid.best_score_,
        4
    )
)


# ============================================================
# 11. GRADIENT BOOSTING OPTIMIZATION
# ============================================================

print("\n" + "=" * 70)
print("GRADIENT BOOSTING OPTIMIZATION")
print("=" * 70)

gb = GradientBoostingClassifier(
    random_state=42
)

gb_params = {

    "n_estimators": [
        100,
        200
    ],

    "learning_rate": [
        0.03,
        0.05,
        0.1
    ],

    "max_depth": [
        2,
        3,
        4
    ]
}


gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=gb_params,
    cv=3,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

gb_grid.fit(
    X_train_selected,
    y_train
)

best_gb = gb_grid.best_estimator_

print(
    "\nBest Gradient Boosting parameters:"
)

print(
    gb_grid.best_params_
)

print(
    "Best CV F1:",
    round(
        gb_grid.best_score_,
        4
    )
)


# ============================================================
# 12. EVALUATE TUNED MODELS
# ============================================================

print("\n" + "=" * 70)
print("TUNED MODEL EVALUATION")
print("=" * 70)


optimized_models = {

    "Optimized Random Forest":
        best_rf,

    "Optimized SVM":
        best_svm,

    "Optimized Gradient Boosting":
        best_gb
}


optimization_results = []


for model_name, model in optimized_models.items():

    print("\n" + "-" * 60)

    print(
        model_name
    )

    print("-" * 60)

    y_pred = model.predict(
        X_test_selected
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    optimization_results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1_Score": f1

    })

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1 Score : {f1:.4f}"
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )


# ============================================================
# 13. OPTIMIZATION RESULTS
# ============================================================

optimization_df = pd.DataFrame(
    optimization_results
)

print("\n" + "=" * 70)
print("OPTIMIZED MODEL COMPARISON")
print("=" * 70)

print(
    optimization_df.to_string(
        index=False
    )
)


# ============================================================
# 14. SAVE RESULTS
# ============================================================

optimization_path = os.path.join(
    RESULT_DIR,
    "optimized_model_comparison.csv"
)

optimization_df.to_csv(
    optimization_path,
    index=False
)

print(
    "\nSaved:",
    optimization_path
)


# ============================================================
# 15. SAVE OPTIMIZED MODELS
# ============================================================

joblib.dump(
    best_rf,
    os.path.join(
        MODEL_DIR,
        "optimized_random_forest.pkl"
    )
)

joblib.dump(
    best_svm,
    os.path.join(
        MODEL_DIR,
        "optimized_svm.pkl"
    )
)

joblib.dump(
    best_gb,
    os.path.join(
        MODEL_DIR,
        "optimized_gradient_boosting.pkl"
    )

)

joblib.dump(
    label_encoder,
    os.path.join(
        MODEL_DIR,
        "label_encoder_step5.pkl"
    )
)


# ============================================================
# 16. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 70)

importance_df = pd.DataFrame({

    "feature": selected_features,

    "importance":
        best_rf.feature_importances_

})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print(
    importance_df.to_string(
        index=False
    )
)


importance_path = os.path.join(
    RESULT_DIR,
    "random_forest_feature_importance.csv"
)

importance_df.to_csv(
    importance_path,
    index=False
)

print(
    "\nSaved:",
    importance_path
)


# ============================================================
# 17. SAVE BEST MODEL
# ============================================================

best_model_name = (
    optimization_df
    .sort_values(
        "F1_Score",
        ascending=False
    )
    .iloc[0]["Model"]
)

print(
    "\nBest model based on test F1:",
    best_model_name
)


if best_model_name == "Optimized Random Forest":

    best_model = best_rf

elif best_model_name == "Optimized SVM":

    best_model = best_svm

else:

    best_model = best_gb


best_model_path = os.path.join(
    MODEL_DIR,
    "best_optimized_model.pkl"
)

joblib.dump(
    best_model,
    best_model_path
)

print(
    "Saved best model:",
    best_model_path
)


# ============================================================
# 18. SAVE TEST DATA WITH SELECTED FEATURES
# ============================================================

test_features_path = os.path.join(
    RESULT_DIR,
    "selected_test_features.csv"
)

X_test_selected.to_csv(
    test_features_path,
    index=False
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 5 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nSelected features:",
    len(selected_features)
)

print(
    "Models optimized:",
    len(optimized_models)
)

print(
    "\nGenerated results:"
)

print(
    " - feature_scores.csv"
)

print(
    " - selected_features.csv"
)

print(
    " - optimized_model_comparison.csv"
)

print(
    " - random_forest_feature_importance.csv"
)

print(
    "\nGenerated models:"
)

print(
    " - optimized_random_forest.pkl"
)

print(
    " - optimized_svm.pkl"
)

print(
    " - optimized_gradient_boosting.pkl"
)

print(
    " - best_optimized_model.pkl"
)

print(
    "\nNext step:"
)

print(
    "STEP 6 - Model Explainability & Fault Prediction"
)

EDGE AI BEARING FAULT DETECTION
STEP 5: HYPERPARAMETER OPTIMIZATION

Loading engineered features...
Dataset shape: (2300, 26)

Number of classes: 10
Classes: ['Ball_007_1', 'Ball_014_1', 'Ball_021_1', 'IR_007_1', 'IR_014_1', 'IR_021_1', 'Normal_1', 'OR_007_6_1', 'OR_014_6_1', 'OR_021_6_1']

Training samples: 1840
Testing samples: 460

FEATURE SELECTION

Feature ranking:
          feature     f_score       p_value
           log_sd 5799.495504  0.000000e+00
          log_rms 5790.912599  0.000000e+00
               sd 5548.274482  0.000000e+00
              rms 5540.693328  0.000000e+00
     log_variance 4805.507913  0.000000e+00
   log_rms_energy 4802.593444  0.000000e+00
 log_peak_to_peak 4282.380221  0.000000e+00
         variance 4214.639412  0.000000e+00
       rms_energy 4212.823364  0.000000e+00
log_absolute_peak 4080.736354  0.000000e+00
     peak_to_peak 3174.129247  0.000000e+00
              min 3114.841493  0.000000e+00
    absolute_peak 3003.380944  0.000000e+00
           

In [8]:
# ============================================================
# EDGE AI BEARING FAULT DETECTION
# STEP 6: MODEL EXPLAINABILITY & FAULT PREDICTION
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = (
    "/workspaces/edge-ai-bearing-fault-detection"
)

DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "engineered_bearing_features.csv"
)

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "best_optimized_model.pkl"
)

FEATURE_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "optimization",
    "selected_features.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "results",
    "explainability"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 2. HEADER
# ============================================================

print("=" * 70)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 6: MODEL EXPLAINABILITY & FAULT PREDICTION")
print("=" * 70)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading engineered dataset...")

df = pd.read_csv(
    DATA_PATH
)

print(
    "Dataset shape:",
    df.shape
)


# ============================================================
# 4. LOAD SELECTED FEATURES
# ============================================================

print("\nLoading selected features...")

selected_features_df = pd.read_csv(
    FEATURE_PATH
)

selected_features = (
    selected_features_df[
        "feature"
    ].tolist()
)

print(
    "Selected features:",
    len(selected_features)
)

for feature in selected_features:
    print(
        " -",
        feature
    )


# ============================================================
# 5. PREPARE FEATURES AND TARGET
# ============================================================

TARGET = "fault"

X = df[
    selected_features
]

y = df[
    TARGET
]


# ============================================================
# 6. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(
    y
)

print(
    "\nFault classes:"
)

for number, class_name in enumerate(
    label_encoder.classes_
):

    print(
        f"{number} -> {class_name}"
    )


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        random_state=42,
        stratify=y_encoded
    )
)


# ============================================================
# 8. LOAD BEST MODEL
# ============================================================

print("\nLoading optimized model...")

model = joblib.load(
    MODEL_PATH
)

print(
    "Model loaded successfully."
)


# ============================================================
# 9. PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("FAULT PREDICTION")
print("=" * 70)

y_pred = model.predict(
    X_test
)


# ============================================================
# 10. MODEL PERFORMANCE
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"\nTest Accuracy: {accuracy:.4f}"
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)


# ============================================================
# 11. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

display.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title(
    "Best Model - Bearing Fault Confusion Matrix"
)

plt.tight_layout()

confusion_path = os.path.join(
    RESULT_DIR,
    "best_model_confusion_matrix.png"
)

plt.savefig(
    confusion_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(
    "Saved:",
    confusion_path
)


# ============================================================
# 12. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)


# Tree-based models expose feature_importances_
# SVM models generally do not.

if hasattr(
    model,
    "feature_importances_"
):

    importance_values = (
        model.feature_importances_
    )

    importance_df = pd.DataFrame({

        "feature":
            selected_features,

        "importance":
            importance_values

    })

    importance_df = (
        importance_df
        .sort_values(
            by="importance",
            ascending=False
        )
    )

    print(
        importance_df.to_string(
            index=False
        )
    )

    importance_path = os.path.join(
        RESULT_DIR,
        "feature_importance.csv"
    )

    importance_df.to_csv(
        importance_path,
        index=False
    )

    print(
        "\nSaved:",
        importance_path
    )

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    plot_df = (
        importance_df
        .head(15)
        .sort_values(
            "importance"
        )
    )

    plt.figure(
        figsize=(10, 7)
    )

    plt.barh(
        plot_df["feature"],
        plot_df["importance"]
    )

    plt.xlabel(
        "Importance"
    )

    plt.ylabel(
        "Feature"
    )

    plt.title(
        "Top Bearing Fault Prediction Features"
    )

    plt.tight_layout()

    importance_plot_path = os.path.join(
        RESULT_DIR,
        "feature_importance.png"
    )

    plt.savefig(
        importance_plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved:",
        importance_plot_path
    )


else:

    print(
        "The selected model does not expose "
        "tree-based feature_importances_."
    )


# ============================================================
# 13. SHAP EXPLAINABILITY
# ============================================================

print("\n" + "=" * 70)
print("SHAP EXPLAINABILITY")
print("=" * 70)

try:

    import shap

    print(
        "SHAP version:",
        shap.__version__
    )

    # Use a sample to keep SHAP computation manageable

    X_shap = X_test.copy()

    if len(X_shap) > 200:

        X_shap = X_shap.sample(
            200,
            random_state=42
        )

    # Tree models

    if hasattr(
        model,
        "feature_importances_"
    ):

        explainer = shap.TreeExplainer(
            model
        )

        shap_values = explainer.shap_values(
            X_shap
        )

        # SHAP summary plot

        plt.figure()

        shap.summary_plot(
            shap_values,
            X_shap,
            show=False
        )

        shap_summary_path = os.path.join(
            RESULT_DIR,
            "shap_summary_plot.png"
        )

        plt.tight_layout()

        plt.savefig(
            shap_summary_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()

        print(
            "Saved:",
            shap_summary_path
        )

        # SHAP bar plot

        plt.figure()

        shap.summary_plot(
            shap_values,
            X_shap,
            plot_type="bar",
            show=False
        )

        shap_bar_path = os.path.join(
            RESULT_DIR,
            "shap_feature_importance.png"
        )

        plt.tight_layout()

        plt.savefig(
            shap_bar_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()

        print(
            "Saved:",
            shap_bar_path
        )

    else:

        print(
            "SHAP TreeExplainer is not used "
            "because the selected model is not "
            "a tree-based estimator."
        )


except ImportError:

    print(
        "SHAP is not installed."
    )

    print(
        "Install it using:"
    )

    print(
        "pip install shap"
    )


# ============================================================
# 14. SAVE PREDICTION RESULTS
# ============================================================

prediction_results = pd.DataFrame({

    "actual_fault":
        label_encoder.inverse_transform(
            y_test
        ),

    "predicted_fault":
        label_encoder.inverse_transform(
            y_pred
        )

})


# Add confidence/probability when supported

if hasattr(
    model,
    "predict_proba"
):

    probabilities = model.predict_proba(
        X_test
    )

    prediction_results[
        "prediction_confidence"
    ] = probabilities.max(
        axis=1
    )


prediction_path = os.path.join(
    RESULT_DIR,
    "fault_predictions.csv"
)

prediction_results.to_csv(
    prediction_path,
    index=False
)

print(
    "\nSaved:",
    prediction_path
)


# ============================================================
# 15. SAMPLE PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE FAULT PREDICTION")
print("=" * 70)

sample = X_test.iloc[
    [0]
]

actual_class = label_encoder.inverse_transform(
    [y_test[0]]
)[0]

predicted_class = label_encoder.inverse_transform(
    model.predict(sample)
)[0]

print(
    "\nActual fault:",
    actual_class
)

print(
    "Predicted fault:",
    predicted_class
)

if hasattr(
    model,
    "predict_proba"
):

    probability = (
        model.predict_proba(sample)
        .max()
    )

    print(
        f"Confidence: {probability:.2%}"
    )


# ============================================================
# 16. SAVE SAMPLE PREDICTION
# ============================================================

sample_output = sample.copy()

sample_output[
    "actual_fault"
] = actual_class

sample_output[
    "predicted_fault"
] = predicted_class

sample_path = os.path.join(
    RESULT_DIR,
    "sample_fault_prediction.csv"
)

sample_output.to_csv(
    sample_path,
    index=False
)

print(
    "\nSaved:",
    sample_path
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 6 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nModel accuracy:",
    f"{accuracy:.4f}"
)

print(
    "\nGenerated explainability files:"
)

print(
    " - best_model_confusion_matrix.png"
)

print(
    " - feature_importance.csv"
)

print(
    " - feature_importance.png"
)

print(
    " - shap_summary_plot.png"
)

print(
    " - shap_feature_importance.png"
)

print(
    " - fault_predictions.csv"
)

print(
    " - sample_fault_prediction.csv"
)

print(
    "\nNext step:"
)

print(
    "STEP 7 - Edge AI / Real-Time Fault Prediction"
)

EDGE AI BEARING FAULT DETECTION
STEP 6: MODEL EXPLAINABILITY & FAULT PREDICTION

Loading engineered dataset...
Dataset shape: (2300, 26)

Loading selected features...
Selected features: 15
 - log_sd
 - log_rms
 - sd
 - rms
 - log_variance
 - log_rms_energy
 - log_peak_to_peak
 - variance
 - rms_energy
 - log_absolute_peak
 - peak_to_peak
 - min
 - absolute_peak
 - max
 - mean

Fault classes:
0 -> Ball_007_1
1 -> Ball_014_1
2 -> Ball_021_1
3 -> IR_007_1
4 -> IR_014_1
5 -> IR_021_1
6 -> Normal_1
7 -> OR_007_6_1
8 -> OR_014_6_1
9 -> OR_021_6_1

Loading optimized model...
Model loaded successfully.

FAULT PREDICTION



Test Accuracy: 0.9348

Classification Report:
              precision    recall  f1-score   support

  Ball_007_1       0.93      0.91      0.92        46
  Ball_014_1       0.93      0.80      0.86        46
  Ball_021_1       0.86      0.78      0.82        46
    IR_007_1       0.98      1.00      0.99        46
    IR_014_1       1.00      1.00      1.00        46
    IR_021_1       0.92      1.00      0.96        46
    Normal_1       1.00      1.00      1.00        46
  OR_007_6_1       1.00      1.00      1.00        46
  OR_014_6_1       0.77      0.87      0.82        46
  OR_021_6_1       0.98      0.98      0.98        46

    accuracy                           0.93       460
   macro avg       0.94      0.93      0.93       460
weighted avg       0.94      0.93      0.93       460

Saved: /workspaces/edge-ai-bearing-fault-detection/results/explainability/best_model_confusion_matrix.png

FEATURE IMPORTANCE
The selected model does not expose tree-based feature_importances_.


In [10]:
# ============================================================
# EDGE AI BEARING FAULT DETECTION
# STEP 7: EDGE AI / REAL-TIME FAULT PREDICTION
# ============================================================

import os
import sys
import time
import joblib
import numpy as np
import pandas as pd

from datetime import datetime


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = (
    "/workspaces/edge-ai-bearing-fault-detection"
)

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "best_optimized_model.pkl"
)

FEATURE_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "optimization",
    "selected_features.csv"
)

ENGINEERED_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "engineered_bearing_features.csv"
)

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "results",
    "edge_ai"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)


# ============================================================
# 2. HEADER
# ============================================================

print("=" * 70)
print("EDGE AI BEARING FAULT DETECTION")
print("STEP 7: EDGE AI / REAL-TIME FAULT PREDICTION")
print("=" * 70)


# ============================================================
# 3. CHECK REQUIRED FILES
# ============================================================

print("\nChecking required files...")

required_files = {
    "Model": MODEL_PATH,
    "Selected features": FEATURE_PATH,
    "Engineered dataset": ENGINEERED_DATA_PATH
}

for name, path in required_files.items():

    if os.path.exists(path):

        print(
            f"[OK] {name}: {path}"
        )

    else:

        print(
            f"[ERROR] Missing {name}: {path}"
        )

        sys.exit(1)


# ============================================================
# 4. LOAD MODEL
# ============================================================

print("\nLoading trained model...")

model = joblib.load(
    MODEL_PATH
)

print(
    "Model loaded successfully."
)


# ============================================================
# 5. LOAD FEATURE LIST
# ============================================================

selected_features_df = pd.read_csv(
    FEATURE_PATH
)

selected_features = (
    selected_features_df[
        "feature"
    ].tolist()
)

print(
    "\nFeatures expected by model:",
    len(selected_features)
)

for feature in selected_features:

    print(
        " -",
        feature
    )


# ============================================================
# 6. LOAD LABEL ENCODER
# ============================================================

label_encoder_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "label_encoder_step5.pkl"
)

if os.path.exists(
    label_encoder_path
):

    label_encoder = joblib.load(
        label_encoder_path
    )

    print(
        "\nLabel encoder loaded."
    )

else:

    print(
        "\nWarning: label encoder not found."
    )

    label_encoder = None


# ============================================================
# 7. LOAD ENGINEERED DATA
# ============================================================

df = pd.read_csv(
    ENGINEERED_DATA_PATH
)

print(
    "\nEngineered dataset:",
    df.shape
)


# ============================================================
# 8. VERIFY FEATURES
# ============================================================

missing_features = [
    feature
    for feature in selected_features
    if feature not in df.columns
]

if missing_features:

    print(
        "\nMissing model features:"
    )

    for feature in missing_features:

        print(
            " -",
            feature
        )

    sys.exit(1)


# ============================================================
# 9. REAL-TIME PREDICTION FUNCTION
# ============================================================

def predict_bearing_fault(
    feature_data
):
    """
    Predict bearing fault from
    engineered vibration features.
    """

    # Convert input to DataFrame

    input_df = pd.DataFrame(
        [feature_data]
    )

    # Make sure feature order matches
    # training order

    input_df = input_df[
        selected_features
    ]

    # Prediction

    prediction = model.predict(
        input_df
    )[0]

    # Decode class

    if label_encoder is not None:

        predicted_fault = (
            label_encoder
            .inverse_transform(
                [prediction]
            )[0]
        )

    else:

        predicted_fault = prediction


    # Confidence

    confidence = None

    if hasattr(
        model,
        "predict_proba"
    ):

        probabilities = (
            model
            .predict_proba(
                input_df
            )[0]
        )

        confidence = float(
            np.max(
                probabilities
            )
        )


    return (
        predicted_fault,
        confidence
    )


# ============================================================
# 10. SELECT SAMPLE DATA
# ============================================================

print("\n" + "=" * 70)
print("REAL-TIME INFERENCE TEST")
print("=" * 70)

# Use first sample as simulated sensor input

sample = df[
    selected_features
].iloc[0]


sample_features = (
    sample.to_dict()
)


# ============================================================
# 11. RUN PREDICTION
# ============================================================

start_time = time.perf_counter()

predicted_fault, confidence = (
    predict_bearing_fault(
        sample_features
    )
)

end_time = time.perf_counter()


inference_time_ms = (
    end_time - start_time
) * 1000


# ============================================================
# 12. DISPLAY PREDICTION
# ============================================================

print(
    "\nPrediction result:"
)

print(
    "Predicted fault:",
    predicted_fault
)

if confidence is not None:

    print(
        f"Confidence: "
        f"{confidence:.2%}"
    )

print(
    f"Inference time: "
    f"{inference_time_ms:.3f} ms"
)


# ============================================================
# 13. MAINTENANCE ALERT
# ============================================================

print(
    "\nMaintenance status:"
)

if confidence is not None:

    if confidence >= 0.90:

        print(
            "HIGH-CONFIDENCE PREDICTION"
        )

    elif confidence >= 0.70:

        print(
            "MODERATE-CONFIDENCE PREDICTION"
        )

    else:

        print(
            "LOW-CONFIDENCE PREDICTION"
        )

else:

    print(
        "Confidence unavailable."
    )


# ============================================================
# 14. BATCH REAL-TIME SIMULATION
# ============================================================

print("\n" + "=" * 70)
print("BATCH EDGE INFERENCE SIMULATION")
print("=" * 70)

number_of_samples = min(
    20,
    len(df)
)

inference_results = []

for index in range(
    number_of_samples
):

    sample_features = (
        df[
            selected_features
        ].iloc[index]
        .to_dict()
    )

    start = time.perf_counter()

    predicted_fault, confidence = (
        predict_bearing_fault(
            sample_features
        )
    )

    end = time.perf_counter()

    latency_ms = (
        end - start
    ) * 1000

    result = {

        "sample_id":
            index,

        "predicted_fault":
            predicted_fault,

        "confidence":
            confidence,

        "inference_time_ms":
            latency_ms,

        "timestamp":
            datetime.now().isoformat()

    }

    inference_results.append(
        result
    )


# ============================================================
# 15. CREATE INFERENCE DATAFRAME
# ============================================================

inference_df = pd.DataFrame(
    inference_results
)

print(
    inference_df.to_string(
        index=False
    )
)


# ============================================================
# 16. SAVE INFERENCE RESULTS
# ============================================================

inference_path = os.path.join(
    RESULT_DIR,
    "edge_inference_results.csv"
)

inference_df.to_csv(
    inference_path,
    index=False
)

print(
    "\nSaved:",
    inference_path
)


# ============================================================
# 17. LATENCY ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("EDGE INFERENCE PERFORMANCE")
print("=" * 70)

average_latency = (
    inference_df[
        "inference_time_ms"
    ].mean()
)

maximum_latency = (
    inference_df[
        "inference_time_ms"
    ].max()
)

minimum_latency = (
    inference_df[
        "inference_time_ms"
    ].min()
)

print(
    f"Average latency: "
    f"{average_latency:.3f} ms"
)

print(
    f"Minimum latency: "
    f"{minimum_latency:.3f} ms"
)

print(
    f"Maximum latency: "
    f"{maximum_latency:.3f} ms"
)


# ============================================================
# 18. SAVE PERFORMANCE METRICS
# ============================================================

performance_df = pd.DataFrame({

    "metric": [
        "average_latency_ms",
        "minimum_latency_ms",
        "maximum_latency_ms",
        "samples_tested"
    ],

    "value": [
        average_latency,
        minimum_latency,
        maximum_latency,
        number_of_samples
    ]

})

performance_path = os.path.join(
    RESULT_DIR,
    "edge_inference_performance.csv"
)

performance_df.to_csv(
    performance_path,
    index=False
)

print(
    "\nSaved:",
    performance_path
)


# ============================================================
# 19. SAVE MODEL METADATA
# ============================================================

metadata = {

    "project":
        "Edge AI Bearing Fault Detection",

    "model":
        os.path.basename(
            MODEL_PATH
        ),

    "number_of_features":
        len(selected_features),

    "features":
        selected_features,

    "inference_samples":
        number_of_samples,

    "average_latency_ms":
        average_latency,

    "minimum_latency_ms":
        minimum_latency,

    "maximum_latency_ms":
        maximum_latency,

    "created_at":
        datetime.now().isoformat()

}

metadata_df = pd.DataFrame(
    [metadata]
)

metadata_path = os.path.join(
    RESULT_DIR,
    "edge_model_metadata.csv"
)

metadata_df.to_csv(
    metadata_path,
    index=False
)

print(
    "Saved:",
    metadata_path
)


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 7 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nThe model can now:"
)

print(
    "1. Accept engineered vibration features"
)

print(
    "2. Predict a bearing fault"
)

print(
    "3. Return prediction confidence"
)

print(
    "4. Measure inference latency"
)

print(
    "5. Simulate repeated edge inference"
)

print(
    "\nGenerated files:"
)

print(
    " - edge_inference_results.csv"
)

print(
    " - edge_inference_performance.csv"
)

print(
    " - edge_model_metadata.csv"
)


print("=" * 70)

EDGE AI BEARING FAULT DETECTION
STEP 7: EDGE AI / REAL-TIME FAULT PREDICTION

Checking required files...
[OK] Model: /workspaces/edge-ai-bearing-fault-detection/models/best_optimized_model.pkl
[OK] Selected features: /workspaces/edge-ai-bearing-fault-detection/results/optimization/selected_features.csv
[OK] Engineered dataset: /workspaces/edge-ai-bearing-fault-detection/data/processed/engineered_bearing_features.csv

Loading trained model...
Model loaded successfully.

Features expected by model: 15
 - log_sd
 - log_rms
 - sd
 - rms
 - log_variance
 - log_rms_energy
 - log_peak_to_peak
 - variance
 - rms_energy
 - log_absolute_peak
 - peak_to_peak
 - min
 - absolute_peak
 - max
 - mean

Label encoder loaded.

Engineered dataset: (2300, 26)

REAL-TIME INFERENCE TEST

Prediction result:
Predicted fault: Ball_007_1
Inference time: 5.763 ms

Maintenance status:
Confidence unavailable.

BATCH EDGE INFERENCE SIMULATION
 sample_id predicted_fault confidence  inference_time_ms                 